# Rilevamento di Attacchi Zero-Day tramite Deep Anomaly Learning
Implementazione di una pipeline di Anomaly Detection basata su Autoencoder per il dataset UNSW-NB15.

### Cella 1: Setup e Import delle Librerie
Importiamo i framework necessari per la manipolazione dei dati e la costruzione della rete neurale in PyTorch.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

print("Librerie importate correttamente. PyTorch version:", torch.__version__)

Librerie importate correttamente. PyTorch version: 2.12.0+cu130


### Cella 2: Caricamento Dati e Pulizia Iniziale


In [ ]:
train_path = "/dataset/UNSW_NB15_training-set.csv"
test_path = "/dataset/UNSW_NB15_testing-set.csv"

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print("Caricamento dataset in corso...")

df_train.fillna(df_train.mean(numeric_only=True), inplace=True)
df_test.fillna(df_test.mean(numeric_only=True), inplace=True)
df_train.drop_duplicates(inplace=True)

print("Operazioni di pulizia (imputazione medie e drop duplicati) completate.")

Caricamento dataset in corso...
Operazioni di pulizia (imputazione medie e drop duplicati) completate.


### Cella 3: Ingegnerizzazione delle Feature
Applichiamo il One-Hot Encoding per le variabili categoriali, isoliamo ESCLUSIVAMENTE il traffico benigno per l'addestramento e applichiamo lo Scaling Min-Max.

In [ ]:
print("Esecuzione One-Hot Encoding...")

categorical_cols = ['proto', 'service', 'state']

df_train = pd.get_dummies(df_train, columns=categorical_cols)
df_test = pd.get_dummies(df_test, columns=categorical_cols)

print("Allineamento colonne...")

df_train, df_test = df_train.align(df_test, join='outer', axis=1, fill_value=0)

print("Separazione label e isolamento traffico benigno...")

# Filtriamo SOLO le righe benigne per il training
df_train_benign = df_train[df_train['label'] == 0]

# Estraiamo le label 
y_train = df_train_benign['label']
y_test = df_test['label']

# Drop delle colonne inutili/leak
drop_cols = ['label', 'attack_cat', 'id']

x_train = df_train_benign.drop(columns=drop_cols, errors='ignore')
x_test = df_test.drop(columns=drop_cols, errors='ignore')

print("Scaling...")

x_train_log = np.log1p(x_train)
x_test_log = np.log1p(x_test)

scaler = MinMaxScaler()
x_train_scaled = scaler.fit_transform(x_train_log)
x_test_scaled = scaler.transform(x_test_log)

print(x_train_scaled.shape)
print(x_test_scaled.shape)

print("OK pipeline completata")

Esecuzione One-Hot Encoding...
Allineamento colonne...
Separazione label e isolamento traffico benigno...
Scaling...
(56000, 196)
(82332, 196)
OK pipeline completata


### Cella 4: Definizione dell'Architettura Autoencoder


In [ ]:
input_dim = x_train_scaled.shape[1]
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.2),
            nn.Linear(64, 32)
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, input_dim)   
        )
    def forward(self, x):
        return self.decoder(self.encoder(x))

autoencoder = Autoencoder(input_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=0.0007)

### Cella 5: Addestramento del Modello
Si istruisce il modello sui dati etichettati come benigni, applicando logiche di Early Stopping personalizzate per prevenire l'overfitting.

In [ ]:
import copy

# Dividiamo il training in train e validation
x_train_split, x_val_split = train_test_split(x_train_scaled, test_size=0.2, random_state=42)

# Creazione dei DataLoader in PyTorch
train_tensor = torch.FloatTensor(x_train_split)
val_tensor = torch.FloatTensor(x_val_split)

batch_size = 128
train_loader = DataLoader(TensorDataset(train_tensor, train_tensor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(val_tensor, val_tensor), batch_size=batch_size, shuffle=False)

print("Inizio addestramento...")
epochs = 100
patience = 10
best_val_loss = float('inf')
patience_counter = 0
best_model_weights = None

for epoch in range(epochs):
    # --- FASE DI TRAINING ---
    autoencoder.train()
    train_loss = 0.0
    for batch_x, _ in train_loader:
        optimizer.zero_grad()
        outputs = autoencoder(batch_x)
        loss = criterion(outputs, batch_x)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch_x.size(0)
    train_loss /= len(train_loader.dataset)

    # --- FASE DI VALIDATION ---
    autoencoder.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_x, _ in val_loader:
            outputs = autoencoder(batch_x)
            loss = criterion(outputs, batch_x)
            val_loss += loss.item() * batch_x.size(0)
    val_loss /= len(val_loader.dataset)

    

    if epoch == 0 or epoch % 5 == 0:
        print(f"Epoch {epoch}/{epochs} - loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

    # --- EARLY STOPPING ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Salva i migliori pesi
        best_model_weights = copy.deepcopy(autoencoder.state_dict())
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping attivato all'epoca {epoch+1}!")
            break

# Ripristina i migliori pesi trovati prima di fermarsi
if best_model_weights is not None:
    autoencoder.load_state_dict(best_model_weights)
    print("Migliori pesi ripristinati con successo.")


Inizio addestramento...
Epoch 0/100 - loss: 0.0001 - val_loss: 0.0000
Epoch 5/100 - loss: 0.0001 - val_loss: 0.0000
Epoch 10/100 - loss: 0.0001 - val_loss: 0.0000
Epoch 15/100 - loss: 0.0001 - val_loss: 0.0000
Epoch 20/100 - loss: 0.0001 - val_loss: 0.0000
Epoch 25/100 - loss: 0.0001 - val_loss: 0.0000
Early stopping attivato all'epoca 26!
Migliori pesi ripristinati con successo.


### Cella 6: Calcolo Dinamico della Soglia (Scoring)


In [ ]:
autoencoder.eval()

# SOLO BENIGNI per il training reconstruction error
x_train_benign = x_train_scaled[y_train == 0]

x_train_tensor = torch.FloatTensor(x_train_benign)

with torch.no_grad():
    pred_train = autoencoder(x_train_tensor).numpy()

mse_train = np.mean((x_train_benign - pred_train)**2, axis=1)

# Soglia τ (per anomaly detection)
tau = np.percentile(mse_train, 99)

print(f"Soglia dinamica (tau): {tau:.5f}")

Soglia dinamica (tau): 0.00028


### Cella 7: Valutazione e Calcolo dei KPI sul Test Set
Inferenza sui dati di test (che contengono classi di attacco miste) e misurazione oggettiva delle metriche architetturali.

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score
import torch
import numpy as np

autoencoder.eval()

x_test_tensor = torch.FloatTensor(x_test_scaled)

with torch.no_grad():
    pred_test = autoencoder(x_test_tensor).numpy()


mse_test = np.mean((x_test_scaled - pred_test)**2, axis=1)
# classificazione binaria
y_pred = (mse_test > tau).astype(int)

# confusion matrix
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

# metriche
auc = roc_auc_score(y_test, mse_test)
detection_rate = TP / (TP + FN) if (TP + FN) > 0 else 0
false_alarm_rate = FP / (FP + TN) if (FP + TN) > 0 else 0

print("--- Metriche di Validazione ---")
print(f"AUC:              {auc:.4f}")
print(f"Detection Rate:   {detection_rate * 100:.2f}% (target >95%)")
print(f"False Alarm Rate: {false_alarm_rate * 100:.2f}% (target <5%)")

--- Metriche di Validazione ---
AUC:              0.9408
Detection Rate:   74.93% (target >95%)
False Alarm Rate: 4.57% (target <5%)


### Cella 8: Simulazione 


In [91]:
import random
import time

# Prendo 5 benigni e 5 attacchi a caso
benign_indices = np.where(y_test == 0)[0]
attack_indices = np.where(y_test == 1)[0]
selected_indices = list(random.sample(benign_indices.tolist(), 5)) + list(random.sample(attack_indices.tolist(), 5))
random.shuffle(selected_indices)

print("Avvio simulazione con campioni misti...\n")
for idx in selected_indices:
    sample = x_test_scaled[idx].reshape(1, -1)
    sample_tensor = torch.FloatTensor(sample)
    
    start_time = time.time()
    with torch.no_grad():
        reconstruction = autoencoder(sample_tensor).numpy()
    error = np.mean((sample - reconstruction)**2)
    latency_ms = (time.time() - start_time) * 1000
    
    true_label = "Attacco" if y_test.iloc[idx] == 1 else "Benigno"
    status = "[!!] MINACCIA ZERO-DAY" if error > tau else "[OK] Normale"

    if (true_label == "Attacco" and error > tau) or (true_label == "Benigno" and error < tau):
        print(f"Campione {idx} | Reale: {true_label} | Previsto: {status} | Latenza: {latency_ms:.2f}ms | Previsione CORRETTA")
    else:
        print(f"Campione {idx} | Reale: {true_label} | Previsto: {status} | Latenza: {latency_ms:.2f}ms | Previsione ERRATA")


Avvio simulazione con campioni misti...

Campione 48098 | Reale: Attacco | Previsto: [OK] Normale | Latenza: 4.77ms | Previsione ERRATA
Campione 17084 | Reale: Attacco | Previsto: [!!] MINACCIA ZERO-DAY | Latenza: 4.10ms | Previsione CORRETTA
Campione 39840 | Reale: Benigno | Previsto: [OK] Normale | Latenza: 3.00ms | Previsione CORRETTA
Campione 52698 | Reale: Attacco | Previsto: [!!] MINACCIA ZERO-DAY | Latenza: 4.32ms | Previsione CORRETTA
Campione 25653 | Reale: Benigno | Previsto: [OK] Normale | Latenza: 4.43ms | Previsione CORRETTA
Campione 30263 | Reale: Benigno | Previsto: [OK] Normale | Latenza: 3.88ms | Previsione CORRETTA
Campione 46184 | Reale: Attacco | Previsto: [OK] Normale | Latenza: 7.13ms | Previsione ERRATA
Campione 78909 | Reale: Benigno | Previsto: [OK] Normale | Latenza: 3.25ms | Previsione CORRETTA
Campione 59610 | Reale: Attacco | Previsto: [!!] MINACCIA ZERO-DAY | Latenza: 7.83ms | Previsione CORRETTA
Campione 38741 | Reale: Benigno | Previsto: [OK] Normale | L

# Cella 9 - Salvataggio del modello

In [92]:
import joblib
import json

# Pesi
torch.save(autoencoder.state_dict(), 'autoencoder_zeroday.pth')

# Scaler 
joblib.dump(scaler, 'scaler_zeroday.joblib')

# KPI in JSON 
results_summary = {
    "model":          "Autoencoder UNSW-NB15",
    "auc":            round(float(auc), 4),
    "detection_rate": round(float(detection_rate) * 100, 2),
    "false_alarm_rate": round(float(false_alarm_rate) * 100, 2),
    "threshold_tau":  round(float(tau), 6),
    "confusion_matrix": {"TN": int(TN), "FP": int(FP), "FN": int(FN), "TP": int(TP)},
    "kpi_targets": {
        "AUC_target":       ">0.90",
        "DR_target":        ">95%",
        "FAR_target":       "<5%",
    }
}

with open('results_summary.json', 'w', encoding='utf-8') as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False)


print(" DATI SALVATI \n")

print("  autoencoder_zeroday.pth -> pesi del modello")
print("  scaler_zeroday.joblib   -> scaler Min-Max")
print("  results_summary.json    -> KPI summary")
print("=" * 50)
print(json.dumps(results_summary["kpi_targets"], indent=4, ensure_ascii=False))

 DATI SALVATI 

  autoencoder_zeroday.pth -> pesi del modello
  scaler_zeroday.joblib   -> scaler Min-Max
  results_summary.json    -> KPI summary
{
    "AUC_target": ">0.90",
    "DR_target": ">95%",
    "FAR_target": "<5%"
}


# Cella 10 - Preparazione dashboard opertiva

In [93]:
# Salvo i dati in un file JSON

import json
import numpy as np

# Campiona 400 sample misti dal test set
np.random.seed(42)
n_sim = min(400, len(mse_test))
sim_idx = np.random.permutation(len(mse_test))[:n_sim]

# Crea il dizionario con i dati
dashboard_data = {
    "mse":    [round(float(x), 7) for x in mse_test[sim_idx]],
    "labels": [int(x) for x in y_test.values[sim_idx]],
    "tau":    round(float(tau), 7),
    "auc":    round(float(auc), 4),
    "dr":     round(float(detection_rate) * 100, 2),
    "far":    round(float(false_alarm_rate) * 100, 2),
}

# Salva i dati in un file JSON
with open("dashboard_data.json", "w", encoding="utf-8") as f:
    json.dump(dashboard_data, f)

print(f"Dati esportati con successo in 'dashboard_data.json'.")
print("Per avviare la dashboard Streamlit digitare: \n \n streamlit run dashboard.py \n \n !!! Dashboard link: http://localhost:8501")

Dati esportati con successo in 'dashboard_data.json'.
Per avviare la dashboard Streamlit digitare: 
 
 streamlit run dashboard.py 
 
 !!! Dashboard link: http://localhost:8501
